# Web API

# Hva er et API?

For å forstå hvordan moderne programvare samhandler, må vi først definere begrepet **API**, som står for *Application Programming Interface* (på norsk: programmeringsgrensesnitt). Et API fungerer som en kontrakt eller en bruksanvisning som beskriver nøyaktig hvordan en programvarekomponent skal brukes av en annen. Det definerer hvilke funksjoner som er tilgjengelige, hvilke data man må sende inn, og hva man kan forvente å få i retur.

Når vi jobber lokalt i Python, møter vi API-er hele tiden uten at vi nødvendigvis tenker over det. Når du bruker et bibliotek som `pandas` eller `numpy`, er det bibliotekets API som bestemmer hvilke kommandoer du må skrive for å for eksempel lese en CSV-fil eller beregne et gjennomsnitt.

---

## Web API:

Et **Web API** er en spesialisert form for API som gjør ressurser tilgjengelige over internett. I stedet for at koden din snakker med en pakke som ligger installert på din egen maskin, sender programmet ditt en forespørsel over nettverket til en ekstern server.


I vår kontekst dreier dette seg om en strukturert måte for kommunikasjon mellom to parter:
1. **Klienten:** Dette er din egen maskin eller Python-programmet du skriver.
2. **Serveren:** Dette er en ekstern datakilde, for eksempel hos Statistisk sentralbyrå (SSB) eller Eurostat.

Gjennom Web API-et kan klienten "be om" spesifikke datasett, og serveren svarer med å sende de forespurte dataene tilbake i et format maskinen forstår (ofte JSON eller XML). Dette gjør at vi kan automatisere henting av data uten å måtte laste ned filer manuelt via en nettleser.

---


Når en bruker Web API-er går man fra å jobbe med statiske filer på egen PC, til å kunne hente og behandle enorme mengder oppdatert data fra hele verden.

## REST API: Standardarkitekturen for datautveksling

Selv om "Web API" er et bredt begrep, vil du i de aller fleste tilfeller støte på en spesiell og svært populær variant som kalles et **REST API** (eller *RESTful API*). 

REST er et akronym for *REpresentational State Transfer*. Selv om navnet kan virke tungt og teknisk, beskriver det i bunn og grunn et sett med standardiserte kjøreregler for hvordan klienter og servere skal utveksle informasjon over internett. Når et API er bygget i henhold til disse reglene, sier vi at det er *RESTful*. 

For oss som skal skrive Python-kode for å hente data, innebærer dette at vi forholder oss til fire sentrale prinsipper:

### 1. HTTP-metoder (Verbet)
Når klienten (vårt program) ber om en ressurs, skjer dette gjennom standardiserte HTTP-metoder. Metoden fungerer som et "verb" som forteller serveren hva vi ønsker å gjøre. Den desidert vanligste for dataanalyse er **GET**, som betyr "hent disse dataene til meg". Hvis vi derimot skulle sende eller opprette ny data på serveren, ville vi brukt metoden **POST**.

### 2. Endepunkter (Substantivet)
I et REST API har hver unike ressurs sin helt egen nettadresse (URL). I API-terminologi kaller vi en slik adresse for et **endepunkt** (*endpoint*). Endepunktet fungerer som et "substantiv" i forespørselen. Vil vi for eksempel ha befolkningsstatistikk fra SSB, retter vi Python-koden vår mot det spesifikke endepunktet (URL-en) som representerer akkurat dette datasettet.

![](https://images.ctfassets.net/vwq10xzbe6iz/5sBH4Agl614xM7exeLsTo7/9e84dce01735f155911e611c42c9793f/rest-api.png)


### 3. Representasjon av data
Når vi "treffer" et endepunkt med en forespørsel, sender ikke serveren hele den underliggende databasen sin til oss. I stedet genererer den en *representasjon* av de forespurte dataene der og da. I moderne REST API-er er denne representasjonen nesten utelukkende formatert som **JSON** (JavaScript Object Notation). Dette er et strukturert tekstformat som er svært enkelt for datamaskiner å parse, og som Python elegant kan gjøre om til lister, ordbøker eller en Pandas DataFrame.

### 4. Tilstandsløshet (Statelessness)
Et av de mest fundamentale prinsippene i REST, er at kommunikasjonen er **tilstandsløs** (*stateless*). Det betyr at serveren *ikke* husker deg fra forrige forespørsel. Det finnes ingen "pågående sesjon", og du er ikke vedvarende innlogget mellom handlinger. Fordi serveren er "glemsk", må *hver eneste forespørsel* du sender inneholde absolutt alt webserveren trenger

## Bruk
For å ta i bruk et REST-API bør vi altså kunne litt om:
* HTTP-protokollen
* Oppbygging av en URL
* JSON-formatet, og jsonstat-formatet med `pyjstat`
* Bruk av `requests` biblioteket til å sende http-spørringer

Når det er på plass kan man ta i bruk mange kule ressurser!
Se feks:
* [https://free-apis.github.io/#/categories](https://free-apis.github.io/#/categories)
* [https://github.com/public-apis/public-apis](https://github.com/public-apis/public-apis)

# SSB
SSB er i gang med å lage et nytt grensesnitt for sitt webapi høsten 2025.
Det ser stort sett ut til å være ferdig, men noen støtteverktøy for å bygge spørringer er borte og det hele virker litt vanskelig. Vi ville tippe at SSB vil komme til å lage nye versjoner av tidligere støtteverktøy (API-konsoll) som gjør det enklere å «skreddersy» større og mer kompliserte spørringer.

Vi skal se litt på hvordan ssb sitt webapi er satt sammen, og på noen måter å bruke det på. Vi gjør dette i økende vanskelighetsgrad:
1. Gjør utvalg/filtrer i statistikkbank og bruk ferdig lenke med `GET`
2. Gjør endringer manuelt med `GET`eller `POST` spørring
3. Bygge spørring direkte i pxweb


### Spotpris metaller
Vi tar utgangspunkt i tabell 07199, spotpris metaller [](https://www.ssb.no/statbank/table/07199). Letteste måte å hente data programmatisk er å gjøre et utvalg i statistikkbanken og trykke kopiere `GET`-urlen du finner under «Lagre» og «API-spørring» på venstre side. 
![image.png](lagresporring3.png)

Dersom vi velger alle metaller unntatt aluminium, og krysser av for alle måneder, får vi denne `GET`-urlen:
```
url = "https://data.ssb.no/api/pxwebapi/v2/tables/07199/data?lang=no&outputFormat=json-stat2&valuecodes[ContentsCode]=Kopper,Nikkel,Sink,Bly,Gull,Silver&valuecodes[Tid]=*&heading=Tid&stub=ContentsCode"
```

Denne kan vi nå ganske enkelt bruke til å hente dataene våre inn til pandas:

In [39]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
from pyjstat import pyjstat

url = "https://data.ssb.no/api/pxwebapi/v2/tables/07199/data?lang=no&outputFormat=json-stat2&valuecodes[ContentsCode]=Kopper,Nikkel,Sink,Bly,Gull,Silver&valuecodes[Tid]=*&heading=Tid&stub=ContentsCode"

dataset = pyjstat.Dataset.read(url)
print("Tabell", dataset["label"])
df = dataset.write("dataframe")
df

Tabell 07199: Spotpris metaller, etter statistikkvariabel og måned


,statistikkvariabel,måned,value
0,Kopper (kr per tonn),1990M01,15357.2
1,Kopper (kr per tonn),1990M02,15147.5
2,Kopper (kr per tonn),1990M03,17265.8
3,Kopper (kr per tonn),1990M04,17474.2
4,Kopper (kr per tonn),1990M05,17728.6
...,...,...,...
2593,Sølv (kr per unse),2025M09,427.1
2594,Sølv (kr per unse),2025M10,483.9
2595,Sølv (kr per unse),2025M11,587.8
2596,Sølv (kr per unse),2025M12,714.5


Vi velger nå å formatere url-en litt: vi begynner hver parameter på en ny linje ved å bruke `\` som linjebrudd.
Det lar oss kikke litt på de ulike parameterene

In [118]:
url = "https://data.ssb.no/api/pxwebapi/v2/tables/07199/data?lang=no&outputFormat=json-stat2\
&valuecodes[ContentsCode]=Kopper,Nikkel,Sink,Bly,Gull,Silver\
&valuecodes[Tid]=*\
&heading=Tid&stub=ContentsCode"

Vi har her 2 variabler: Type metall og måned. Type metall er gitt som en kommaseparert liste etter `valueCodes[ContentsCode]=`.
Forhar vi bare et stjernesymbol `*`. Disse parameterene angir filtrreringen vi gjorde i statistikkbanken. Dersom vi vil gjøre et annet utvalg kan vi endre på verdiene her. Nedenfor følger en oversikt

### Oversikt over verdi-operatorer i SSB PxWebApi v2

| Operatør | Eksempel | Beskrivelse |
| :--- | :--- | :--- |
| `*` eller `??` | `valueCodes[Tid]=*` | **Alt:** Henter absolutt alle tilgjengelige verdier for variabelen. |
| `kode*` | `valueCodes[Region]=03*` | **Starter med:** Henter alle koder som begynner på spesifisert tekst (f.eks. alle i Oslo). |
| `?` | `valueCodes[Region]=030?` | **Ett tegn:** Erstatter nøyaktig ett tegn (f.eks. henter 0301, 0302). |
| `top(n)` | `valueCodes[Tid]=top(5)` | **Nyeste:** Henter de *n* nyeste (siste) verdiene i listen. |
| `bottom(n)` | `valueCodes[Tid]=bottom(1)` | **Eldste:** Henter de *n* eldste (første) verdiene i listen. |
| `from(kode)` | `valueCodes[Tid]=from(2020)` | **Fra og med:** Henter alle koder fra og med valgt verdi og utover. |
| `range(x, y)` | `valueCodes[Konto]=range(01, 05)` | **Intervall:** Henter alle verdier i et lukket intervall mellom x og y. |
| `,` | `0301, 1103` | **Liste:** Henter spesifikke koder separert med komma. |

In [50]:
# Eksempel: Alle metaller som begynner på "S" og tidsrom fra 90-tallet:

url = "https://data.ssb.no/api/pxwebapi/v2/tables/07199/data?lang=no&outputFormat=json-stat2\
&valuecodes[ContentsCode]=S*\
&valuecodes[Tid]=199*\
&heading=Tid&stub=ContentsCode"

dataset = pyjstat.Dataset.read(url)
df = dataset.write("dataframe")
df

,statistikkvariabel,måned,value
0,Sink (kr per tonn),1990M01,8463.3
1,Sink (kr per tonn),1990M02,8970.2
2,Sink (kr per tonn),1990M03,10534.6
3,Sink (kr per tonn),1990M04,10939.5
4,Sink (kr per tonn),1990M05,11366.6
...,...,...,...
235,Sølv (kr per unse),1999M08,41.1
236,Sølv (kr per unse),1999M09,41.1
237,Sølv (kr per unse),1999M10,42.1
238,Sølv (kr per unse),1999M11,40.8


In [52]:
# Eksempel: Alle metaller, spotpris siden mars 2012

url = "https://data.ssb.no/api/pxwebapi/v2/tables/07199/data?lang=no&outputFormat=json-stat2\
&valuecodes[ContentsCode]=*\
&valuecodes[Tid]=from(2012M03)\
&heading=Tid&stub=ContentsCode"

dataset = pyjstat.Dataset.read(url)
df = dataset.write("dataframe")
df

,statistikkvariabel,måned,value
0,Aluminium (kr per tonn),2012M03,12449.9
1,Aluminium (kr per tonn),2012M04,11730.4
2,Aluminium (kr per tonn),2012M05,11793.9
3,Aluminium (kr per tonn),2012M06,11374.5
4,Aluminium (kr per tonn),2012M07,11405.7
...,...,...,...
1164,Sølv (kr per unse),2025M09,427.1
1165,Sølv (kr per unse),2025M10,483.9
1166,Sølv (kr per unse),2025M11,587.8
1167,Sølv (kr per unse),2025M12,714.5


Det er også av og til mulig å bruke `requests` med `params` argumentetet til å bygge url-strengen

In [56]:
url = "https://data.ssb.no/api/pxwebapi/v2/tables/07199/data?lang=no&outputFormat=json-stat2"
params = {"valuecodes[ContentsCode]": "*", "valuecodes[Tid]": "from(2012M03)"}
resp = requests.get(url, params=params)
print("Status", resp.status_code)
print("URL fra requests:", resp.url)
#URL ser rar ut '[' er feks blitt til %5B
#Dette er url-tegnkoding og blir gjort automatisk av requests

Status 200
URL fra requests: https://data.ssb.no/api/pxwebapi/v2/tables/07199/data?lang=no&outputFormat=json-stat2&valuecodes%5BContentsCode%5D=%2A&valuecodes%5BTid%5D=from%282012M03%29


In [57]:
dataset = pyjstat.Dataset.read(resp.text)
df = dataset.write('dataframe')
df

,statistikkvariabel,måned,value
0,Aluminium (kr per tonn),2012M03,12449.9
1,Aluminium (kr per tonn),2012M04,11730.4
2,Aluminium (kr per tonn),2012M05,11793.9
3,Aluminium (kr per tonn),2012M06,11374.5
4,Aluminium (kr per tonn),2012M07,11405.7
...,...,...,...
1164,Sølv (kr per unse),2025M09,427.1
1165,Sølv (kr per unse),2025M10,483.9
1166,Sølv (kr per unse),2025M11,587.8
1167,Sølv (kr per unse),2025M12,714.5


Dersom vi vil gi en liste av metaller må vi gi den slik:

In [59]:
url = "https://data.ssb.no/api/pxwebapi/v2/tables/07199/data?lang=no&outputFormat=json-stat2"
params = {"valuecodes[ContentsCode]": "Kopper,Nikkel,Sink,Bly,Gull,Silver", 
          "valuecodes[Tid]": "from(2012M03)"}
resp = requests.get(url, params=params)
print("Status", resp.status_code)
print("URL fra requests:", resp.url)
#URL ser rar ut '[' er feks blitt til %5B
#Dette er url-tegnkoding og blir gjort automatisk av requests

Status 200
URL fra requests: https://data.ssb.no/api/pxwebapi/v2/tables/07199/data?lang=no&outputFormat=json-stat2&valuecodes%5BContentsCode%5D=Kopper%2CNikkel%2CSink%2CBly%2CGull%2CSilver&valuecodes%5BTid%5D=from%282012M03%29


In [60]:
dataset = pyjstat.Dataset.read(resp.text)
df = dataset.write('dataframe')

,statistikkvariabel,måned,value
0,Kopper (kr per tonn),2012M03,48144.2
1,Kopper (kr per tonn),2012M04,47323.3
2,Kopper (kr per tonn),2012M05,46539.9
3,Kopper (kr per tonn),2012M06,44795.7
4,Kopper (kr per tonn),2012M07,45910.3
...,...,...,...
997,Sølv (kr per unse),2025M09,427.1
998,Sølv (kr per unse),2025M10,483.9
999,Sølv (kr per unse),2025M11,587.8
1000,Sølv (kr per unse),2025M12,714.5


For store kompliserte spørringer (SSB foreslår over 2200 tegn) burde vi bruke en `POST`-spørring – ssb støtter nemlig begge.
I statistikkbanken kan vi velge om vi vil ha get eller post
![](lagresporring-post.png)

Velger vi "post" kan vi kopiere ut post-url og en data-payload med utdragene vi må legge ved som data i post-spørringen

In [62]:
post_url = "https://data.ssb.no/api/pxwebapi/v2/tables/07199/data?lang=no&outputFormat=json-stat2"

payload = {
  "selection": [
    {
      "variableCode": "ContentsCode",
      "valueCodes": [
        "Kopper",
        "Nikkel",
        "Sink",
        "Bly",
        "Gull",
        "Silver"
      ]
    },
    {
      "variableCode": "Tid",
      "valueCodes": [
        "*"
      ]
    }
  ],
  "placement": {
    "heading": [
      "Tid"
    ],
    "stub": [
      "ContentsCode"
    ]
  }
}



Til en post-spørring må vi altså legge ved noe data, en payload. Denne inneholder eventuelle filtreringer og utdrag vi vil gjøre med samme formatering som tidligere.

In [67]:
#Endre spørring til alle metall
payload["selection"][0]["valueCodes"] = ["*"]

#Endre spørring til siste 24 måneder
payload["selection"][1]["valueCodes"] = ["top(24)"]

#Merk at "*" og top(24) nå må inn i en liste

resp = requests.post(post_url, json=payload)
print("Statuskode", resp.status_code)


Statuskode 200


In [68]:
dataset = pyjstat.Dataset.read(resp.text)
df = dataset.write("dataframe")
df

,statistikkvariabel,måned,value
0,Aluminium (kr per tonn),2024M02,19141.6
1,Aluminium (kr per tonn),2024M03,19232.6
2,Aluminium (kr per tonn),2024M04,19767.3
3,Aluminium (kr per tonn),2024M05,20004.1
4,Aluminium (kr per tonn),2024M06,23552.1
...,...,...,...
163,Sølv (kr per unse),2025M09,427.1
164,Sølv (kr per unse),2025M10,483.9
165,Sølv (kr per unse),2025M11,587.8
166,Sølv (kr per unse),2025M12,714.5


### Gjøre alt gjennom pxweb

- Det er også mulig å holde seg helt utenom statistikkbanken på ssb sine vevsider.
- [](https://data.ssb.no/api/pxwebapi/v2/tables/) gir en oversikt over alle tabellene til ssb
- De fleste nettlesere kan faktisk åpne denne URL-en slik at en kan få en oversikt
- Ved å legge ved url-parameteren `query` kan vi søke i tabellene

In [69]:
px_url = "https://data.ssb.no/api/pxwebapi/v2/tables/"
resp = requests.get(px_url)
tabeller = resp.json()

In [72]:
for tab in tabeller["tables"]:
    print("Tabellno: ", tab["id"], tab["label"])

Tabellno:  13760 13760: Arbeidsstyrken, sysselsatte, arbeidsledige og utførte ukeverk, etter kjønn og alder. Brudd- og sesongjusterte tall 2006M01-2025M12
Tabellno:  14483 14483: Personer, etter arbeidsstyrkestatus, kjønn og alder. Brudd- og sesongjusterte tall 2009K1-2025K4
Tabellno:  13618 13618: Personer, etter arbeidsstyrkestatus, kjønn og alder. Bruddjusterte tall 2009-2025
Tabellno:  13619 13619: Personer, etter arbeidsstyrkestatus, kjønn og alder. Bruddjusterte tall 2009K1-2025K4
Tabellno:  05110 05110: Personer, etter arbeidsstyrkestatus, kjønn og alder 1988K2-2025K4
Tabellno:  05111 05111: Personer, etter arbeidsstyrkestatus, kjønn og alder 1972-2025
Tabellno:  14077 14077: Personer under utdanning siste fire uker, etter kjønn, alder og arbeidsstyrkestatus 2021K1-2025K4
Tabellno:  14090 14090: Personer under utdanning siste fire uker, etter kjønn, alder og arbeidsstyrkestatus 2021-2025
Tabellno:  13784 13784: Personer, etter arbeidsstyrkestatus, kjønn, alder og utdanningsnivå 

URL til neste side med resultater finnes under "page"->"links"

In [75]:
tabeller["page"]["links"]

[{'rel': 'next',
  'hreflang': 'no',
  'href': 'https://data.ssb.no/api/pxwebapi/v2/tables/?lang=no&pagesize=20&pageNumber=2'},
 {'rel': 'last',
  'hreflang': 'no',
  'href': 'https://data.ssb.no/api/pxwebapi/v2/tables/?lang=no&pagesize=20&pageNumber=195'}]

In [76]:
url_neste = tabeller["page"]["links"][0]["href"]
resp = requests.get(url_neste)
tabeller2 = resp.json()


In [77]:
for tab in tabeller2["tables"]:
    print("Tabellno: ", tab["id"], tab["label"])

Tabellno:  13893 13893: Delvis arbeidsledige og undersysselsatte, etter kjønn, alder og utdanningsnivå 2021-2025
Tabellno:  04554 04554: Arbeidsledige og delvis arbeidsledige, etter ønsket arbeidstid per uke. Tilbud av ukeverk (a 37,5 timer) 1996K1-2025K4
Tabellno:  04555 04555: Arbeidsledige og delvis arbeidsledige, etter ønsket arbeidstid per uke. Tilbud av ukeverk (a 37,5 timer) 1996-2025
Tabellno:  13534 13534: Den utvidede arbeidsstyrken, etter kjønn, arbeidsmarkedsstatus og alder 2021K1-2025K4
Tabellno:  13535 13535: Den utvidede arbeidsstyrken, etter kjønn, arbeidsmarkedsstatus og alder 2021-2025
Tabellno:  13583 13583: Personer utenfor arbeidsstyrken, etter kjønn, alder og hovedsakelig virksomhet 2021K1-2025K4
Tabellno:  13584 13584: Personer utenfor arbeidsstyrken, etter kjønn, alder og hovedsakelig virksomhet 2021-2025
Tabellno:  14455 14455: Personer utenfor arbeidsstyrken, etter kjønn, alder, søkeaktivitet og ønske om arbeid 2021K1-2025K4
Tabellno:  14456 14456: Personer ut

### Mer heavy metal 
Vi kan prøve å finnne og hente ut spotprisdataene våres ved å søke og bla i pxweb sitt api. Merk at lenkene her lar seg åpne i nettleseren - det er en god måte å lete i json-strukturene

In [78]:
# Ved å legge ved en query parameter gjør vi et søk i tabellene :)
sokeparams = {"query": "spotpris metaller"} 
resp = requests.get(px_url, params=sokeparams)
resp.raise_for_status()


In [80]:
tabeller = resp.json()
for tab in tabeller["tables"]:
    print("Tabellno: ", tab["id"], tab["label"])

Tabellno:  07199 07199: Spotpris metaller 1990M01-2026M01
Tabellno:  07201 07201: Spotpris metaller 1990-2025


In [82]:
tabell_spotpris = tabeller["tables"][0] # Vi har funnet tabellen vår
for lenke in tabell_spotpris["links"]:
    print("Lenke", lenke["rel"], "\n", lenke["href"])

Lenke self 
 https://data.ssb.no/api/pxwebapi/v2/tables/07199?lang=no
Lenke alternate 
 https://data.ssb.no/api/pxwebapi/v2/tables/07199?lang=en
Lenke metadata 
 https://data.ssb.no/api/pxwebapi/v2/tables/07199/metadata?lang=no
Lenke data 
 https://data.ssb.no/api/pxwebapi/v2/tables/07199/data?lang=no&outputFormat=json-stat2


In [84]:
url_metadata = tabell_spotpris["links"][-2]["href"]
url_data = tabell_spotpris["links"][-1]["href"]

resp = requests.get(url_metadata)
metadata = resp.json()

Nå kan vi prøve å printe ut litt aktuell metadata:

In [113]:
print("Tittel", metadata["label"])
print("Notater fra ssb", metadata["note"])
print("Statistikkvariabler, id", metadata["id"])
print("Tidsspenn, fra:", tabell_spotpris["firstPeriod"], 
      "til: ", tabell_spotpris["lastPeriod"])
print("Metaller", metadata["dimension"]["ContentsCode"]["category"]["index"].keys())

Tittel 07199: Spotpris metaller 1990M01-2026M01
Notater fra ssb ['Kilde: Quandl, som igjen er basert på data fra London Metal Exchange (LME) og andre råvarebørser. Fra og med Juli 2018 er gjennomsnittsprisene i tabellen beregnet på en annen metode enn før Juli 2018']
Statistikkvariabler, id ['ContentsCode', 'Tid']
Tidsspenn, fra: 1990M01 til:  2026M01
Metaller dict_keys(['Aluminium', 'Kopper', 'Nikkel', 'Sink', 'Bly', 'Gull', 'Silver'])


Vi kan hente ut alle mulige verdier for statistikkvariablene eller «dimensjonene» i datasettet:

In [112]:
dimensions = { 
    var_id: list(metadata["dimension"][var_id]["category"]["index"].keys()) 
    for var_id in metadata["id"]
}
# Ikke lett å finne fram i json-stat for et menneske, men her må vi

- Vi har nå hentet frem tabellid, metadata og url for dataene vi skal hente
- Vi må nå bygge spørringen fra bunnen av ved å undersøke dimensjon/metadata
- Vi må altså velge hvilke data det er vi skal spørre etter

In [115]:
url = url_data # funnet fram tidligere

post_payload = {"selection" : [ {"variableCode": dim_id, "valueCodes": dim_vals} 
                                for dim_id, dim_vals in dimensions.items()]
               }

resp = requests.post(url, json=post_payload)
resp.raise_for_status()


In [116]:
ds = pyjstat.Dataset.read(resp.text)
df = ds.write("dataframe")
df

,statistikkvariabel,måned,value
0,Aluminium (kr per tonn),1990M01,9987.9
1,Aluminium (kr per tonn),1990M02,9360.1
2,Aluminium (kr per tonn),1990M03,10287.5
3,Aluminium (kr per tonn),1990M04,9983.1
4,Aluminium (kr per tonn),1990M05,9859.3
...,...,...,...
3026,Sølv (kr per unse),2025M09,427.1
3027,Sølv (kr per unse),2025M10,483.9
3028,Sølv (kr per unse),2025M11,587.8
3029,Sølv (kr per unse),2025M12,714.5


### Oppsummering: SSB

- Vi har hoppet over mange detaljer om hvordan spørringen er oppbygd
- Se [](https://www.ssb.no/api/pxwebapiv2) dersom du trenger å gjøre aggregering, gruppering osv
- Enkleste bruk er å gjøre filtreringer i statistikkbanken, og hente ut get-url derifra

::: {admonition} Oppgåve
Oppgavetekst
:::


::: {hint}
Pass på både dill og dall
:::